In [ ]:
import torch
from transformers import AutoModel, AutoProcessor
from transformers.image_utils import load_image

# load the model and processor
ckpt = "google/siglip2-so400m-patch16-naflex"
model = AutoModel.from_pretrained(ckpt, device_map="auto").eval()
processor = AutoProcessor.from_pretrained(ckpt)


In [ ]:
import torch_xla
import torch_xla.core.xla_model as xm
device = torch_xla.device()

memory_info = xm.get_memory_info(device)

print(memory_info)

model = model.to(device)

In [ ]:
from functools import lru_cache
import imageio
import requests
import numpy as np
import tqdm

@lru_cache(maxsize=1000)
def download_image(image_url: str) -> np.ndarray:
    response = requests.get(image_url, verify=False)
    return imageio.imread(response.content)

In [ ]:
import json
import urllib.request
import zipfile
import os
from collections import defaultdict

ANNOT_URL = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"
ANNOT_ZIP = "annotations_trainval2017.zip"
ANNOT_FILE = "annotations/instances_val2017.json"

if not os.path.exists(ANNOT_FILE):
    print("Downloading annotations ...")
    urllib.request.urlretrieve(ANNOT_URL, ANNOT_ZIP)
    with zipfile.ZipFile(ANNOT_ZIP, 'r') as z:
        z.extract(ANNOT_FILE)
    print("Download and extraction complete.")

with open(ANNOT_FILE) as f:
    coco = json.load(f)

TARGET_CLASSES = ['cat', 'dog', 'car', 'person']
IMAGES_PER_CLASS = 100

cat_name_to_id = {c['name']: c['id'] for c in coco['categories'] if c['name'] in TARGET_CLASSES}
cat_id_to_name = {v: k for k, v in cat_name_to_id.items()}

img_meta = {img['id']: img for img in coco['images']}
candidates = defaultdict(list)

for ann in coco['annotations']:
    cat_id = ann['category_id']
    if cat_id not in cat_id_to_name:
        continue

    cat_name = cat_id_to_name[cat_id]
    img = img_meta[ann['image_id']]
    img_area = img['width'] * img['height']

    if img_area == 0:
        continue

    # [x, y, width, height]
    bbox_area = ann['bbox'][2] * ann['bbox'][3]
    ratio = bbox_area / img_area

    candidates[cat_name].append((ratio, img['file_name']))

BASE_URL = "https://huggingface.co/datasets/merve/coco/resolve/main/val2017/"
category_urls = {}

for cat_name in TARGET_CLASSES:
    cat_candidates = sorted(candidates[cat_name], key=lambda x: x[0], reverse=True)

    seen = set()
    unique_urls = []

    for ratio, fname in cat_candidates:
        if fname not in seen:
            seen.add(fname)
            unique_urls.append(f"{BASE_URL}{fname}")

        if len(unique_urls) == IMAGES_PER_CLASS:
            break

    category_urls[cat_name] = unique_urls



In [ ]:
from concurrent.futures import ThreadPoolExecutor

cat_to_id = {}

urls_to_embed = []
targets_str = []
targets_int = []

for cat_name, urls in category_urls.items():
    for url in urls:
        urls_to_embed.append(url)
        targets_str.append(cat_name)

        if cat_name not in cat_to_id:
            cat_to_id[cat_name] = len(cat_to_id)

        targets_int.append(cat_to_id[cat_name])

pool = ThreadPoolExecutor(16)
images_to_embed = list(tqdm.tqdm(pool.map(download_image, urls_to_embed), total=len(urls_to_embed)))

targets_int = np.array(targets_int)
targets_str = np.array(targets_str)

In [ ]:
inputs = processor(images=images_to_embed, return_tensors="pt").to(model.device)

with torch.no_grad():
    embeddings = model.get_image_features(**inputs).pooler_output

embeddings /= embeddings.norm(dim=-1, keepdim=True)

embeddings.shape

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

embeddings_2d = TSNE(2, metric='cosine').fit_transform(embeddings.cpu().detach().numpy())

fig = plt.figure(figsize=(10, 8))

plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], c=targets_int)
plt.grid()
plt.show()


In [ ]:
# Делаем поисковую систему!

text_requests = ["красная машина"]
text_inputs = processor(text=[x.lower() for x in text_requests], return_tensors="pt").to(model.device)

with torch.no_grad():
    text_embeddings = model.get_text_features(**text_inputs).pooler_output

text_embeddings /= text_embeddings.norm(dim=-1, keepdim=True)

distances = (text_embeddings[:, None, :] * embeddings[None, :, :]).sum(dim=-1)
scores = torch.sigmoid(distances * model.logit_scale.exp() + model.logit_bias)

scores.shape

In [ ]:
topk = torch.topk(scores, 5, dim=-1)

top_indices = topk.indices[0].cpu().detach().numpy()
top_scores = topk.values[0].cpu().detach().numpy()

for index, score in zip(top_indices, top_scores):
    # print(index, score)
    image = images_to_embed[index]
    plt.title(f'Score={score}')
    plt.imshow(image)
    plt.show()


In [ ]:
# Просто мем, задача - запромптить на скор 0.95 или выше

test_image = download_image('https://s2.wine.style/images_gen/216/216069/0_0_695x600.webp')
plt.imshow(test_image)
plt.show()

with torch.no_grad():
    t_out = model(**processor(images=[test_image], text=["бутылка", "вино", "wine"], return_tensors="pt").to(model.device))

In [ ]:
t_out